# Dispersion-Assisted Optical Phase Recovery
**Time-domain Gerchberg-Saxton algorithm â€” end-to-end demo**

Reference: Solli, Gupta & Jalali, *Appl. Phys. Lett.* **95**, 231108 (2009)

This notebook is a self-contained tutorial that works in **Colab or locally**.  
It imports `gs_core` from the project root; in Colab use the install cell below.

In [ ]:
# â”€â”€ Colab setup (skip if running locally) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Clone repo and install deps
    os.system('git clone https://github.com/jalalilab/Dispersion-Assisted-GS-Phase-Recovery /content/repo')
    sys.path.insert(0, '/content/repo')
    os.system('pip install -q numpy matplotlib scipy sympy')
else:
    # Local: add project root to path
    sys.path.insert(0, os.path.join(os.getcwd(), '..'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import welch
import warnings
warnings.filterwarnings('ignore')

from gs_core import (
    disperse, undisperse, apply_amplitude_constraint,
    gs_iteration, retrieve_phase,
    make_qpsk_measurements, make_measurements,
    retrieve_phase_3d, show_transfer_function,
)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print('gs_core loaded successfully.')

---
## Â§1 Â· System overview

The **dispersive Fourier transform (DFT)** maps the *temporal* field through a dispersive medium:

$$E_d(t) = \mathcal{F}^{-1}\!\left[\, \hat{E}(\nu)\cdot H(\nu) \,\right], \qquad H(\nu) = e^{i\pi D\nu^2}$$

A photodiode measures only **intensity** $I(t)=|E_d(t)|^2$, discarding phase.  
Two arms with dispersions $D_1\neq D_2$ provide independent intensity measurements $I_1, I_2$.

The **time-domain GS algorithm** alternates between two constraint projections:

| Step | Operation |
|------|-----------|
| 1 | Disperse current estimate through $D_1$ |
| 2 | Replace amplitude with $\sqrt{I_1}$ (keep phase) |
| 3 | Undisperse back to original domain |
| 4 | Repeat with $D_2$ |

Convergence requires **sufficient dispersion diversity**: $|D_1-D_2| / |D_1| \gtrsim 0.1$ and $|D| \gtrsim 5000$ (normalized units).

## Â§2 Â· Transfer function (SymPy derivation)

In [ ]:
from sympy import symbols, exp, pi, I, latex, simplify, expand_func
from IPython.display import display, Math

nu, D, beta2, L, omega = symbols('nu D beta_2 L omega', real=True)

# GVD phase in terms of angular frequency
phi_gvd = beta2 * L * omega**2 / 2
# Convert to cyclic frequency nu = omega/(2*pi)
phi_gvd_nu = phi_gvd.subs(omega, 2*pi*nu)
H_full = exp(I * phi_gvd_nu)

# Simplified normalized form used in gs_core
H_norm, H_latex = show_transfer_function()

display(Math(r'\text{GVD phase (angular):}\quad \phi(\omega) = ' + latex(phi_gvd)))
display(Math(r'\text{Transfer function:}\quad H(\nu) = ' + H_latex))
print(f'\nNormalized form (used in gs_core): H = {H_norm}')

## Â§3 Â· Single-run QPSK demo

In [ ]:
# Generate synthetic QPSK measurements
data = make_qpsk_measurements(n_symbols=128, snr_db=25.0, D1=-5000, D2=-5750)
I1, I2 = data['I1'], data['I2']
phi_true = data['phi_true']
t = data['t']

# Run GS
phi_est, errors = retrieve_phase(I1, I2, data['D1'], data['D2'], n_iter=50)

# Align global phase offset (GS can only recover phase up to a constant)
offset = np.angle(np.mean(np.exp(1j * (phi_true - phi_est))))
delta  = np.angle(np.exp(1j * (phi_est + offset - phi_true)))
rms    = float(np.sqrt(np.mean(delta**2)))

print(f'RMS phase error: {rms:.4f} rad  ({np.degrees(rms):.2f}Â°)')
print(f'Final GS amplitude error: {errors[-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

N_show = 300
axes[0].plot(t[:N_show], I1[:N_show], label=f'$I_1$  ($D_1$={data["D1"]:.0f})', lw=1.2)
axes[0].plot(t[:N_show], I2[:N_show], label=f'$I_2$  ($D_2$={data["D2"]:.0f})', lw=1.2, alpha=0.8)
axes[0].set_title('Measured intensities')
axes[0].set_xlabel('Normalized time'); axes[0].legend(fontsize=9)

axes[1].plot(t[:N_show], phi_true[:N_show], label='True $\\phi(t)$', lw=1.5)
axes[1].plot(t[:N_show], (phi_est + offset)[:N_show], '--', label='Recovered $\\hat{\\phi}(t)$', lw=1.5)
axes[1].set_title(f'Phase recovery  (RMS = {np.degrees(rms):.1f}Â°)')
axes[1].set_xlabel('Normalized time'); axes[1].set_ylabel('Phase (rad)'); axes[1].legend(fontsize=9)

axes[2].semilogy(errors, 'o-', color='crimson', markersize=4)
axes[2].set_title('GS convergence')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Amplitude error (RMS)')
axes[2].axhline(errors[-1], ls='--', color='gray', alpha=0.6)

plt.suptitle('TD-GS Phase Retrieval â€” QPSK, SNR=25 dB', fontweight='bold')
plt.tight_layout()
plt.savefig('demo_qpsk_single.png', bbox_inches='tight')
plt.show()

## Â§4 Â· Multi-format comparison

GS performance depends on the modulation format.  Constant-envelope signals (QPSK, DPSK) use the `unit_amplitude=True` constraint; variable-envelope signals (OOK, PAM4, STEAM, Soliton) use `unit_amplitude=False`.

In [ ]:
FORMATS = ['OOK', 'PAM4', 'QPSK', 'DPSK', 'STEAM', 'Soliton']
COLORS  = ['#e41a1c','#ff7f00','#377eb8','#4daf4a','#984ea3','#a65628']

results = {}
for fmt in FORMATS:
    d = make_measurements(modulation=fmt, n_symbols=64, snr_db=25.0)
    phi_hat, errs = retrieve_phase(
        d['I1'], d['I2'], d['D1'], d['D2'],
        n_iter=50, unit_amplitude=d['unit_amplitude']
    )
    off = np.angle(np.mean(np.exp(1j * (d['phi_true'] - phi_hat))))
    dlt = np.angle(np.exp(1j * (phi_hat + off - d['phi_true'])))
    rms = float(np.sqrt(np.mean(dlt**2)))
    results[fmt] = {'phi_hat': phi_hat + off, 'phi_true': d['phi_true'],
                    'errors': errs, 'rms': rms, 't': d['t'],
                    'unit_amp': d['unit_amplitude']}
    print(f'{fmt:8s}  unit_amp={str(d["unit_amplitude"]):5s}  RMS={np.degrees(rms):6.2f}Â°  final_err={errs[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
N_show = 200

for ax, fmt, col in zip(axes, FORMATS, COLORS):
    r = results[fmt]
    ax.plot(r['t'][:N_show], r['phi_true'][:N_show], 'k', lw=1.5, label='True')
    ax.plot(r['t'][:N_show], r['phi_hat'][:N_show],  '--', color=col, lw=1.5, label='GS')
    ax.set_title(f'{fmt}   RMS={np.degrees(r["rms"]):.1f}Â°  (UA={r["unit_amp"]})')
    ax.set_xlabel('Normalized time'); ax.set_ylabel('Phase (rad)')
    ax.legend(fontsize=8)

plt.suptitle('TD-GS: Six modulation formats â€” SNR=25 dB, 50 iterations', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('demo_multiformat.png', bbox_inches='tight')
plt.show()

In [ ]:
# Convergence curves for all formats
fig, ax = plt.subplots(figsize=(8, 5))
for fmt, col in zip(FORMATS, COLORS):
    ax.semilogy(results[fmt]['errors'], 'o-', color=col, label=fmt, markersize=3, lw=1.5)
ax.set_title('GS convergence â€” all modulation formats')
ax.set_xlabel('Iteration'); ax.set_ylabel('Amplitude error (RMS)')
ax.legend(ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo_convergence_multiformat.png', bbox_inches='tight')
plt.show()

## Â§5 Â· SNR sweep

How does noise floor affect recovery quality?  Sweep SNR from 5 dB to 40 dB.

In [ ]:
SNR_DB = np.arange(5, 42, 3)
SWEEP_FORMATS = ['QPSK', 'OOK', 'DPSK']
sweep_rms = {fmt: [] for fmt in SWEEP_FORMATS}

for snr in SNR_DB:
    for fmt in SWEEP_FORMATS:
        d = make_measurements(modulation=fmt, n_symbols=64, snr_db=float(snr), rng_seed=7)
        phi_hat, _ = retrieve_phase(
            d['I1'], d['I2'], d['D1'], d['D2'],
            n_iter=50, unit_amplitude=d['unit_amplitude']
        )
        off = np.angle(np.mean(np.exp(1j * (d['phi_true'] - phi_hat))))
        dlt = np.angle(np.exp(1j * (phi_hat + off - d['phi_true'])))
        sweep_rms[fmt].append(np.degrees(np.sqrt(np.mean(dlt**2))))

fig, ax = plt.subplots(figsize=(8, 5))
SWEEP_COLORS = ['#377eb8','#e41a1c','#4daf4a']
for fmt, col in zip(SWEEP_FORMATS, SWEEP_COLORS):
    ax.plot(SNR_DB, sweep_rms[fmt], 'o-', color=col, label=fmt, lw=1.8, markersize=6)
ax.set_title('Phase error vs. SNR â€” TD-GS (50 iterations, $D_1$=âˆ’5000)')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('RMS phase error (Â°)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo_snr_sweep.png', bbox_inches='tight')
plt.show()

## Â§6 Â· Dispersion parameter sweep

The key design constraint: **$|D| \gtrsim 5000$** (normalized) for convergence.  
Below this, $I_1 \approx I_2$ â€” the two measurements carry no independent information.

In [ ]:
D1_values = [-200, -500, -1000, -2000, -3500, -5000, -7000, -10000]
d_rms, d_corr = [], []

for D1_val in D1_values:
    D2_val = D1_val * 1.15  # paper ratio D2/D1 = 1.15
    d = make_measurements(modulation='QPSK', n_symbols=64, snr_db=30.0,
                          D1=float(D1_val), D2=float(D2_val), rng_seed=3)
    phi_hat, _ = retrieve_phase(
        d['I1'], d['I2'], d['D1'], d['D2'],
        n_iter=50, unit_amplitude=True
    )
    off = np.angle(np.mean(np.exp(1j * (d['phi_true'] - phi_hat))))
    dlt = np.angle(np.exp(1j * (phi_hat + off - d['phi_true'])))
    rms = float(np.sqrt(np.mean(dlt**2)))
    # Diversity metric: correlation of I1 and I2 (high = low diversity)
    corr = float(np.corrcoef(d['I1'], d['I2'])[0, 1])
    d_rms.append(np.degrees(rms))
    d_corr.append(corr)
    print(f'D1={D1_val:7.0f}  D2={D2_val:8.1f}  corr(I1,I2)={corr:.3f}  RMS={np.degrees(rms):.2f}Â°')

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
D1_abs = [abs(x) for x in D1_values]
ax1.semilogy(D1_abs, d_rms, 'o-', color='crimson', lw=2, markersize=8, label='RMS phase error (Â°)')
ax2.plot(D1_abs, d_corr, 's--', color='steelblue', lw=1.5, markersize=7, label='corr($I_1$,$I_2$)')
ax1.axvline(5000, ls=':', color='gray', alpha=0.7, label='$|D|=5000$ threshold')
ax1.set_xlabel('$|D_1|$ (normalized)'); ax1.set_ylabel('RMS phase error (Â°)', color='crimson')
ax2.set_ylabel('Correlation $I_1 \\leftrightarrow I_2$', color='steelblue')
ax1.set_title('Dispersion diversity requirement â€” QPSK, SNR=30 dB')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo_dispersion_sweep.png', bbox_inches='tight')
plt.show()

## Â§7 Â· Iteration count convergence study

How many GS iterations are needed?  Check final RMS error vs. iteration count.

In [ ]:
ITER_COUNTS = [1, 5, 10, 20, 30, 50, 75, 100]
iter_rms_qpsk, iter_rms_ook = [], []

d_qpsk = make_measurements('QPSK', n_symbols=64, snr_db=25.0)
d_ook  = make_measurements('OOK',  n_symbols=64, snr_db=25.0)

for n in ITER_COUNTS:
    for d, store, ua in [(d_qpsk, iter_rms_qpsk, True), (d_ook, iter_rms_ook, False)]:
        phi_hat, _ = retrieve_phase(d['I1'], d['I2'], d['D1'], d['D2'],
                                    n_iter=n, unit_amplitude=ua)
        off = np.angle(np.mean(np.exp(1j * (d['phi_true'] - phi_hat))))
        dlt = np.angle(np.exp(1j * (phi_hat + off - d['phi_true'])))
        store.append(np.degrees(np.sqrt(np.mean(dlt**2))))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ITER_COUNTS, iter_rms_qpsk, 'o-', color='#377eb8', lw=2, markersize=7, label='QPSK')
ax.plot(ITER_COUNTS, iter_rms_ook,  's-', color='#e41a1c', lw=2, markersize=7, label='OOK')
ax.axvline(50, ls='--', color='gray', alpha=0.6, label='n=50 default')
ax.set_title('RMS phase error vs. GS iteration count (SNR=25 dB)')
ax.set_xlabel('Number of iterations'); ax.set_ylabel('RMS phase error (Â°)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo_iter_sweep.png', bbox_inches='tight')
plt.show()

## Â§8 Â· BER estimation from recovered phase

For QPSK, bit-error rate can be estimated from the phase error distribution under the assumption that errors follow a Gaussian distribution and a simple threshold detector is used.

In [ ]:
from scipy.special import erfc

def ber_from_phase_rms(rms_rad, n_bits=2):
    """Estimate BER from RMS phase error for QPSK (n_bits=2 bits/symbol).
    Uses Gaussian noise approximation: BER â‰ˆ 0.5 * erfc(pi/(2*sqrt(2)*sigma))."""
    sigma = rms_rad
    return 0.5 * erfc(np.pi / (2 * np.sqrt(2) * sigma + 1e-12))

SNR_BER = np.arange(5, 35, 2)
ber_gs, ber_theory = [], []

for snr in SNR_BER:
    d = make_measurements('QPSK', n_symbols=128, snr_db=float(snr), rng_seed=42)
    phi_hat, _ = retrieve_phase(d['I1'], d['I2'], d['D1'], d['D2'],
                                n_iter=50, unit_amplitude=True)
    off = np.angle(np.mean(np.exp(1j * (d['phi_true'] - phi_hat))))
    dlt = np.angle(np.exp(1j * (phi_hat + off - d['phi_true'])))
    rms = float(np.sqrt(np.mean(dlt**2)))
    ber_gs.append(ber_from_phase_rms(rms))
    # Theoretical QPSK BER: Q(sqrt(2*Eb/N0))
    EbN0 = 10**(snr/10) / 2  # QPSK: Eb/N0 = SNR/2
    ber_theory.append(0.5 * erfc(np.sqrt(EbN0)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(SNR_BER, ber_gs,     'o-', color='#377eb8', lw=2, markersize=7, label='TD-GS estimated BER')
ax.semilogy(SNR_BER, ber_theory, 's--', color='gray',    lw=1.5, markersize=5, label='Theoretical QPSK BER')
ax.set_title('BER estimate from TD-GS recovered phase â€” QPSK')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Bit Error Rate')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('demo_ber_curve.png', bbox_inches='tight')
plt.show()

## Â§9 Â· 3D phase retrieval: signal stack

`retrieve_phase_3d` processes a batch of independent signals (e.g., different wavelength channels or repeated shots) and optionally enforces phase continuity between adjacent rows.

In [ ]:
M_signals = 20
N_samples  = 512
D1_3d, D2_3d = -5000.0, -5750.0

# Build a stack: each row is an independent QPSK signal at a different seed
I1_stack   = np.zeros((M_signals, N_samples))
I2_stack   = np.zeros((M_signals, N_samples))
phi_true_3d = np.zeros((M_signals, N_samples))

for m in range(M_signals):
    d = make_measurements('QPSK', n_symbols=64, sps=8,
                          D1=D1_3d, D2=D2_3d, snr_db=25.0, rng_seed=m)
    n = min(N_samples, len(d['I1']))
    I1_stack[m, :n]    = d['I1'][:n]
    I2_stack[m, :n]    = d['I2'][:n]
    phi_true_3d[m, :n] = d['phi_true'][:n]

phi_3d, errors_3d = retrieve_phase_3d(
    I1_stack, I2_stack, D1_3d, D2_3d,
    n_iter=50, unit_amplitude=True, phase_continuity=True
)

# RMS per row after global-phase alignment
rms_per_row = []
for m in range(M_signals):
    off = np.angle(np.mean(np.exp(1j * (phi_true_3d[m] - phi_3d[m]))))
    dlt = np.angle(np.exp(1j * (phi_3d[m] + off - phi_true_3d[m])))
    rms_per_row.append(np.degrees(np.sqrt(np.mean(dlt**2))))

print(f'Mean RMS over {M_signals} signals: {np.mean(rms_per_row):.2f}Â° Â± {np.std(rms_per_row):.2f}Â°')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(phi_3d, aspect='auto', cmap='RdBu_r',
                    vmin=-np.pi, vmax=np.pi)
axes[0].set_title('Recovered phase stack $\\hat{\\phi}(m,t)$')
axes[0].set_xlabel('Sample index'); axes[0].set_ylabel('Signal index $m$')
plt.colorbar(im, ax=axes[0], label='Phase (rad)')

axes[1].plot(rms_per_row, 'o-', color='crimson', lw=1.5, markersize=6)
axes[1].axhline(np.mean(rms_per_row), ls='--', color='gray', label=f'Mean={np.mean(rms_per_row):.1f}Â°')
axes[1].set_title('RMS phase error per signal')
axes[1].set_xlabel('Signal index $m$'); axes[1].set_ylabel('RMS error (Â°)')
axes[1].legend()

plt.suptitle('3D phase retrieval stack (20 signals)', fontweight='bold')
plt.tight_layout()
plt.savefig('demo_3d_stack.png', bbox_inches='tight')
plt.show()

## Â§10 Â· Physical units mapping

The normalized dispersion $D$ used in `gs_core` maps to physical DCF fiber parameters as follows:

| Physical | Normalized | Notes |
|----------|-----------|-------|
| $D = -695$ ps/nm | $D_{\text{norm}} \approx -5000$ | Paper Fig. 5(a), arm 1 |
| $D = -800$ ps/nm | $D_{\text{norm}} \approx -5750$ | Paper Fig. 5(a), arm 2 |
| $f_s = 1$ GHz | $N = 1024$, $n_{\text{sym}} = 128$ | Sampling rate |

The normalization is $D_{\text{norm}} = D_{\text{phys}} \times (f_s / \Delta f)^2$ where $\Delta f$ is the signal bandwidth.

In [ ]:
# Verify ratio matches paper
D1_phys, D2_phys = -695.0, -800.0  # ps/nm
D1_norm, D2_norm = -5000.0, -5750.0  # normalized

print(f'Physical ratio   D2/D1 = {D2_phys/D1_phys:.4f}')
print(f'Normalized ratio D2/D1 = {D2_norm/D1_norm:.4f}')
print(f'Scale factor:          = {D1_norm/D1_phys:.2f} (norm units per ps/nm)')
print(f'\nConvergence rule: |D_norm| >= 5000  â†”  |D_phys| >= {5000/(D1_norm/D1_phys):.0f} ps/nm')

## Â§11 Â· Summary

| Topic | Key result |
|-------|------------|
| Transfer function | $H(\nu) = e^{i\pi D\nu^2}$ (GVD in normalized frequency) |
| Convergence requirement | $|D_{\text{norm}}| \geq 5000$, $n_{\text{iter}} = 50$ sufficient |
| Best format | QPSK / DPSK (constant envelope + `unit_amplitude=True`) |
| SNR threshold | ~15 dB for $< 10Â°$ RMS error on QPSK |
| Dispersion diversity | $D_2/D_1 \approx 1.15$; higher ratio â†’ faster convergence |
| 3D extension | Slice-by-slice GS + continuity correction works well |

**Next steps:** integrate real measurement data from the lab (provided by Yiming/Callen), add CNN-based phase refinement ([3]), build Colab demo with upload widget.

## §12 · 3D pipe phase retrieval

`retrieve_phase_pipe` processes a **cylindrical signal stack** $\phi(\theta, z, t)$.

Physical picture: fibers, waveguide arrays, or distributed sensors on a cylinder.
- **Axial continuity**: removes z-drift between adjacent cross-sections.
- **Angular continuity**: closes the wrap-around seam at $\theta=0 \leftrightarrow 2\pi$.

Stack shape: `(N_theta, N_z, N_t)` — angular × axial × temporal.


In [ ]:
from gs_core import retrieve_phase_pipe, pipe_surface_plot

N_THETA, N_Z, N_T = 8, 6, 512
D1_p, D2_p = -5000.0, -5750.0

I1_pipe       = np.zeros((N_THETA, N_Z, N_T))
I2_pipe       = np.zeros((N_THETA, N_Z, N_T))
phi_true_pipe = np.zeros((N_THETA, N_Z, N_T))

seed = 0
for i in range(N_THETA):
    for j in range(N_Z):
        d = make_measurements('QPSK', n_symbols=64, sps=8,
                              D1=D1_p, D2=D2_p, snr_db=25.0, rng_seed=seed)
        seed += 1
        n = min(N_T, len(d['I1']))
        I1_pipe[i, j, :n]       = d['I1'][:n]
        I2_pipe[i, j, :n]       = d['I2'][:n]
        phi_true_pipe[i, j, :n] = d['phi_true'][:n]

print(f'Pipe stack: {I1_pipe.shape}  ({N_THETA}θ x {N_Z}z x {N_T}t)')
phi_pipe, errors_pipe = retrieve_phase_pipe(
    I1_pipe, I2_pipe, D1_p, D2_p,
    n_iter=50, unit_amplitude=True,
    angular_continuity=True, axial_continuity=True,
)

rms_pipe = np.zeros((N_THETA, N_Z))
for i in range(N_THETA):
    for j in range(N_Z):
        off = np.angle(np.mean(np.exp(1j*(phi_true_pipe[i,j]-phi_pipe[i,j]))))
        dlt = np.angle(np.exp(1j*(phi_pipe[i,j]+off-phi_true_pipe[i,j])))
        rms_pipe[i,j] = np.degrees(np.sqrt(np.mean(dlt**2)))

print(f'Mean RMS: {rms_pipe.mean():.2f} deg +/- {rms_pipe.std():.2f} deg')


In [ ]:
# Pipe surface visualization
fig, ax3 = pipe_surface_plot(phi_pipe, t_slice=0)
plt.savefig('notebooks/pipe_surface.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(rms_pipe, cmap='hot_r', aspect='auto')
axes[0].set_xlabel('Axial index z'); axes[0].set_ylabel('Angular index theta')
axes[0].set_title('RMS phase error (deg) on pipe surface')
plt.colorbar(im, ax=axes[0], label='deg')

mean_errs = errors_pipe.mean(axis=(0, 1))
axes[1].semilogy(mean_errs, 'o-', color='crimson', markersize=4)
axes[1].set_title('Mean GS convergence across pipe nodes')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Mean amplitude error')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'3D Pipe phase retrieval  ({N_THETA}theta x {N_Z}z)', fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/demo_pipe_rms.png', bbox_inches='tight')
plt.show()


## §13 · CNN phase refinement + real-data upload widget

Two sub-sections:
- **13a** — 1D CNN (PyTorch): input $(I_1, I_2, \hat{\phi}_{GS})$ → output $\hat{\phi}_{refined}$.
- **13b** — File upload: load real `.npy` measurements from Yiming/Callen.

Reference: [3] Neural network enabled time-stretch spectral regression.


In [ ]:
# §13a — 1D CNN phase refinement
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except ImportError:
    print('PyTorch not installed.  Run: pip install torch')
    TORCH_OK = False

if TORCH_OK:
    class PhaseRefineNet(nn.Module):
        """
        1D CNN residual network: (B, 3, N) -> (B, 1, N).
        Channels: [I1, I2, phi_gs].  Preserves temporal length N.
        """
        def __init__(self, ch=32):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv1d(3,  ch, 7, padding=3), nn.ReLU(),
                nn.Conv1d(ch, ch, 5, padding=2), nn.ReLU(),
                nn.Conv1d(ch, ch, 5, padding=2), nn.ReLU(),
                nn.Conv1d(ch, ch, 3, padding=1), nn.ReLU(),
            )
            self.head = nn.Conv1d(ch, 1, 1)
            self.skip = nn.Conv1d(3,  1, 1)   # residual: passes phi_gs through

        def forward(self, x):
            return self.head(self.enc(x)) + self.skip(x)

    net = PhaseRefineNet()
    print(f'PhaseRefineNet: {sum(p.numel() for p in net.parameters()):,} params')

    # Build training batch (32 QPSK signals at SNR=20 dB)
    def make_batch(n=32, N_t=512, snr=20.0):
        X_list, Y_list = [], []
        for i in range(n):
            d = make_measurements('QPSK', n_symbols=64, sps=8,
                                  D1=-5000., D2=-5750., snr_db=snr, rng_seed=i)
            phi_gs, _ = retrieve_phase(d['I1'], d['I2'], d['D1'], d['D2'],
                                       n_iter=50, unit_amplitude=True)
            nt = min(N_t, len(d['I1']))
            X_list.append([d['I1'][:nt], d['I2'][:nt], phi_gs[:nt]])
            Y_list.append(d['phi_true'][:nt])
        X = torch.tensor(np.array(X_list), dtype=torch.float32)
        Y = torch.tensor(np.array(Y_list)[:, None, :], dtype=torch.float32)
        return X, Y

    print('Building training batch...')
    X_tr, Y_tr = make_batch()

    opt = optim.Adam(net.parameters(), lr=1e-3)
    losses = []
    for ep in range(40):
        net.train(); opt.zero_grad()
        loss = nn.MSELoss()(net(X_tr), Y_tr)
        loss.backward(); opt.step()
        losses.append(float(loss))
        if ep % 10 == 0:
            print(f'  epoch {ep:3d}  loss={float(loss):.5f}')

    # Evaluate on fresh test sample
    net.eval()
    with torch.no_grad():
        d_te = make_measurements('QPSK', n_symbols=64, sps=8,
                                 D1=-5000., D2=-5750., snr_db=20.0, rng_seed=999)
        phi_gs_te, _ = retrieve_phase(d_te['I1'], d_te['I2'],
                                      d_te['D1'], d_te['D2'],
                                      n_iter=50, unit_amplitude=True)
        nt = min(512, len(d_te['I1']))
        X_te = torch.tensor(
            np.stack([d_te['I1'][:nt], d_te['I2'][:nt], phi_gs_te[:nt]])[None],
            dtype=torch.float32)
        phi_cnn_te = net(X_te).squeeze().numpy()

    def rms_deg(phi_hat, phi_ref):
        off = np.angle(np.mean(np.exp(1j*(phi_ref - phi_hat))))
        dlt = np.angle(np.exp(1j*(phi_hat + off - phi_ref)))
        return np.degrees(np.sqrt(np.mean(dlt**2)))

    rgs  = rms_deg(phi_gs_te[:nt],  d_te['phi_true'][:nt])
    rcnn = rms_deg(phi_cnn_te,       d_te['phi_true'][:nt])
    print(f'GS  RMS: {rgs:.2f} deg')
    print(f'CNN RMS: {rcnn:.2f} deg  (improvement: {rgs-rcnn:+.2f} deg)')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    t_show = np.linspace(0,1,nt)
    axes[0].plot(t_show, d_te['phi_true'][:nt], 'k', lw=1.5, label='True')
    axes[0].plot(t_show, phi_gs_te[:nt],  '--', color='steelblue', lw=1.5, label=f'GS  {rgs:.1f}°')
    axes[0].plot(t_show, phi_cnn_te,       '-.',  color='crimson',   lw=1.5, label=f'CNN {rcnn:.1f}°')
    axes[0].set_title('GS vs CNN phase recovery'); axes[0].legend(fontsize=9)
    axes[0].set_xlabel('Normalized time'); axes[0].set_ylabel('Phase (rad)')

    axes[1].semilogy(losses, 'o-', color='steelblue', markersize=4)
    axes[1].set_title('CNN training loss'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE loss'); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('notebooks/demo_cnn_compare.png', bbox_inches='tight')
    plt.show()


In [ ]:
# §13b — Real data upload widget
# Provide .npy files from Yiming (yimingz0416@g.ucla.edu) or Callen (cmacphee@g.ucla.edu)

import os

def load_lab_data(I1_path=None, I2_path=None):
    """
    Load real lab measurements.

    Format: .npy float arrays, shape (N,) — intensity vs. time sample.

    Parameters
    ----------
    I1_path, I2_path : str or None
        Local file paths.  If None and in Colab, opens upload dialog.
        If None and not in Colab, falls back to synthetic QPSK data.

    Returns
    -------
    dict: I1, I2, N, [phi_true if synthetic]
    """
    IN_COLAB = 'google.colab' in __import__('sys').modules

    if I1_path is not None and I2_path is not None:
        I1 = np.load(I1_path).ravel().astype(float)
        I2 = np.load(I2_path).ravel().astype(float)
        print(f'Loaded: I1={I1.shape}, I2={I2.shape}')
        return {'I1': I1, 'I2': I2, 'N': min(len(I1), len(I2))}

    if IN_COLAB:
        from google.colab import files
        import io
        print('Upload I1.npy  (arm 1, dispersion D1):')
        up1 = files.upload()
        print('Upload I2.npy  (arm 2, dispersion D2):')
        up2 = files.upload()
        I1 = np.load(io.BytesIO(list(up1.values())[0])).ravel().astype(float)
        I2 = np.load(io.BytesIO(list(up2.values())[0])).ravel().astype(float)
        return {'I1': I1, 'I2': I2, 'N': min(len(I1), len(I2))}

    # Fallback: synthetic
    print('[DEMO MODE] No file paths provided.  Using synthetic QPSK data.')
    print('To use real data: load_lab_data("I1.npy", "I2.npy")')
    d = make_measurements('QPSK', n_symbols=128, snr_db=25.0, rng_seed=77)
    return {'I1': d['I1'], 'I2': d['I2'], 'N': len(d['I1']),
            'phi_true': d['phi_true'], '_synthetic': True}


# ─── Run on lab data (swap in real file paths once data is provided) ──────
# lab = load_lab_data('data/I1_arm1.npy', 'data/I2_arm2.npy')
lab = load_lab_data()

D1_lab, D2_lab = -5000.0, -5750.0   # physical: -695/-800 ps/nm  (paper values)

phi_lab, errs_lab = retrieve_phase(
    lab['I1'], lab['I2'], D1_lab, D2_lab,
    n_iter=50, unit_amplitude=True,
)

N_show = min(300, lab['N'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(lab['I1'][:N_show], lw=1.2, label='$I_1$ (D1)')
axes[0].plot(lab['I2'][:N_show], lw=1.2, label='$I_2$ (D2)', alpha=0.8)
title_tag = ' [SYNTHETIC DEMO]' if lab.get('_synthetic') else ' [LAB DATA]'
axes[0].set_title('Measurement intensities' + title_tag)
axes[0].set_xlabel('Sample index'); axes[0].legend()

axes[1].plot(phi_lab[:N_show], lw=1.5, color='steelblue', label='GS recovered')
if 'phi_true' in lab:
    off = np.angle(np.mean(np.exp(1j*(lab['phi_true'][:N_show]-phi_lab[:N_show]))))
    axes[1].plot(lab['phi_true'][:N_show], '--', color='crimson',
                 alpha=0.7, label='True (synthetic)')
    axes[1].legend()
axes[1].set_title('Recovered phase'); axes[1].set_xlabel('Sample'); axes[1].set_ylabel('rad')

plt.suptitle('Lab data pipeline (GS phase retrieval)', fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/demo_lab_pipeline.png', bbox_inches='tight')
plt.show()
print('Next: email yimingz0416@g.ucla.edu / cmacphee@g.ucla.edu for real I1.npy, I2.npy')
